In [1]:
import numpy as np, math as math
from decimal import Decimal, getcontext

### (a)

In [2]:
x = 10**8
y = 1.0

z_128 = (np.longdouble(x) + np.longdouble(y)) - np.longdouble(x)
z_64 = (np.double(x) + np.double(y)) - np.double(x)
z_32 = (np.single(x) + np.single(y)) - np.single(x)
print(z_128, z_64, z_32)

1.0 1.0 0.0


The expected result here is z = y = 1. Addition is not commutative in finite-precision binary arithmetic. Single precision floats are only accurate to the 7th digit, and as (x + y) is 10^8 +1  = 100000001, the last digit falls outside the accuracy range. The result is that the calculation becomes (x + y) = x ->  x - x = 0. This is not an issue for double precision and above, as these data types use more bits and have a decimal accuracy which captures the addition of y to x.

### (b)

In [6]:
a, b = 0.1 + 0.2, 0.3 
print(a == b, a - b < 1e-16)

False True


The reason these values don't add up is because none of the numbers can be represented exactly in e.g. double precision floats. For a decimal fraction to be exactly representable in binary, it must be expressible as p/q, where both p and q are integers composed solely of powers of 2. Numbers like 0.1, 0.2, and 0.3 can't be expressed this way, so they are stored as approximations.
To circumvent this issue, an epsilon greater than the precision of the data types which are compared can be used, as seen in the code snippet.

### (c)

In [22]:
x, y, delta = 1.0, 1 + 1e6 * 1e-16, 1e-16
for _ in range(1_000_000): x += delta
print(x==y)

False


What's happening here is that 10^-16 is on the limit of double float precision. Similarly to (a), when it it added to a (comparatively) large number such as x = 1.0, the spacing of decimal places makes it so it falls *just* outside the machine precision. Therefore, when adding small numbers to larger numbers, summing instances of small numbers first is good practice, and in this case solves the problem:

In [23]:
delta, tot = 1e-16, 0.0
for _ in range(1_000_000): tot += delta
x = 1.0 + tot
print(x == y)

True


### (d)

In [ ]:
f_170, f_171 = np.double(1), np.double(1)
for n in range(1, 171): f_170 *= n
for n in range(1, 172): f_171 *= n
print(f_170, f_171)
print(f'Max representable val: {np.finfo(np.double).max}')


7.257415615307994e+306 inf
Max representable val: 1.7976931348623157e+308


C:\Users\knuth\AppData\Local\Temp\ipykernel_11752\135662502.py:3: RuntimeWarning: overflow encountered in scalar multiply
  for n in range(1, 172): f_171 *= n


The maximum stored value with double precision floating point is roughly 1e+308. As seen in the printout, 170! comes very close to this value, while 171! causes data overflow because it exceeds it. One strategy to fix this is to manually set a decimal precision to avoid overflow:

In [49]:
getcontext().prec = 310
f_170, f_171 = Decimal(1), Decimal(1)
for n in range(1, 171): f_170 *= n
for n in range(1, 172): f_171 *= n
print(f'{f_171 - f_170:.5e}')

1.23376e+309


### (e)

In [5]:
a, b = np.double(1.000000000001), np.double(1.0)
direct, conjugate = np.sqrt(a) - np.sqrt(b), (a-b)/(np.sqrt(a) + np.sqrt(b))
d = 1e-12 / (np.sqrt(a) + np.sqrt(b))
print(direct, conjugate, d)

5.000444502911705e-13 5.000444502910455e-13 4.99999999999875e-13


The problem here is the possibility of catastrophic cancellation: when square rooting a and b, a is very close to machine precision. As previously mentioned, adding up smaller numbers is generally better, and thus using the conjugate rule to rephrase such that the square roots are added should increase precision. But, since a - b is already very small, it experiences catastrophic cancellation as well. If the difference d between a and b is known, you can evaluate $\sqrt{a + d} - \sqrt{b}$ directly, which is accurate to the 11th digit, compared to 4 in the other methods (by Taylor exansion of $\sqrt{1+x}$, we know that the expression evaluates to 5e-13 + $O$(e-24)). This is a slight reformulation of the problem, but is a valid approach for e.g. root finding methods.